# 심정지 조기경보 — 베이스라인 파이프라인 (4단계)

**이 노트북 하나로 전체 흐름을 이해**할 수 있게 만들었습니다. 각 단계마다 *왜 하는지*와
*나온 숫자를 어떻게 읽는지*를 설명합니다.

| 단계 | 내용 | 나오는 것 |
|---|---|---|
| 1 | EDA 및 데이터 구조 분석 | 분포·결측·궤적 그래프 |
| 2 | Feature Engineering | 윈도우 통계 + 개인 기저선 이탈 피처 |
| 3 | LightGBM / XGBoost / CatBoost 베이스라인 | 모델 비교표 + PR/ROC |
| 4 | 하이퍼파라미터 튜닝 및 앙상블 | 튜닝 결과 + 앙상블 성능 |

---

## 먼저: 우리가 푸는 문제가 뭔가?

**입력**: 환자의 활력징후(맥박·혈압·체온·산소포화도·호흡수) 시계열
**출력**: "앞으로 H시간 안에 심정지가 올 확률"
**차별점**: 정확도가 아니라 **오경보를 줄이는 것** + **왜 위험한지 설명**

> ⚠️ **지금 쓰는 데이터**: PhysioNet Challenge 2019는 **패혈증(sepsis)** 데이터입니다.
> 우리 주제(심정지)와 다르지만 "활력징후 → 임박한 악화"라는 **구조가 같아** 파이프라인을
> 실데이터로 검증하는 **연습용 프록시**입니다. 진짜 심정지 데이터(경북대)는 본선에서 받습니다.

## 지표 읽는 법 (이게 제일 중요)

| 지표 | 의미 | 왜 보나 |
|---|---|---|
| **AUPRC** | 정밀도-재현율 곡선 아래 면적 | **우리 핵심 지표.** 희귀사건에서 "알람이 울렸을 때 진짜일 확률"을 반영 |
| ROC-AUC | 일반적 분류 성능 | 희귀사건에선 **높게 나와도 쓸모없을 수 있음** (함정) |
| 민감도@95%특이도 | 오경보를 5%로 묶었을 때 잡아내는 비율 | 운영 관점 |
| **알람수/100** | 100개 윈도우당 울리는 알람 수 | **낮을수록 좋음.** alarm fatigue 직결 |
| lead-time | 사건 몇 시간 전에 경보했나 | 조기경보의 본질 |

**AUPRC 판단 기준**: 양성비율(base rate)과 비교하세요.
양성이 1%인데 AUPRC가 0.01이면 **랜덤과 같음**(=실패). 0.05면 5배 좋은 것.

---
## 0. 설정

In [ ]:
import sys, warnings
from pathlib import Path

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# ===== 설정 =====
DATA_DIR  = "/workspace/training_setA"
MAX_FILES = 2000     # None = 전체 (많을수록 좋지만 느림)
HORIZON   = 6        # 예측 지평(시간). 1h면 양성이 너무 희박해 실패함(3단계에서 확인)
SEED      = 42
# ================

VITALS = ["pulse", "sbp", "dbp", "temperature", "spo2", "resp_rate"]
print("repo:", REPO, "\ndata:", DATA_DIR)

---
# 1단계. EDA 및 데이터 구조 분석

**목적**: 모델을 만들기 전에 데이터가 어떻게 생겼는지 파악. 결측이 심한 변수, 이상값,
이벤트 환자 비율을 모르면 나중에 나온 숫자를 해석할 수 없습니다.

## 1-1. 로드 + 데이터 구조

In [ ]:
from vitals_data import cohort_from_challenge2019

cohort = cohort_from_challenge2019(DATA_DIR, max_files=MAX_FILES)

n_pat = cohort.vitals["patient_id"].nunique()
n_evt = int(cohort.events["arrest_hour"].notna().sum())
print(f"환자 {n_pat}명 | 이벤트 발생 {n_evt}명 ({n_evt/n_pat:.1%}) | 대조군 {n_pat-n_evt}명")
print(f"활력징후 기록 {len(cohort.vitals):,}행 (환자당 시간별 1행)\n")

print("[구조] cohort.vitals — 환자별 시점별 활력징후")
display(cohort.vitals.head())
print("\n[구조] cohort.events — 환자별 이벤트 발생 시각(arrest_hour). NaN=대조군")
display(cohort.events.head())

**읽는 법**: `vitals`는 (환자, 시각) 단위 long 테이블입니다. `hour`는 입원 후 경과시간.
`events`의 `arrest_hour`가 그 환자의 사건 발생 시각이고, NaN이면 사건이 없던 대조군입니다.

## 1-2. 기술통계 + 결측

In [ ]:
print("[기술통계]")
display(cohort.vitals[VITALS].describe().T.round(1))

miss = cohort.vitals[VITALS].isna().mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 3.5))
miss.plot.barh(ax=ax, color="indianred")
ax.set_xlabel("fraction missing"); ax.set_title("Missing rate per vital")
for i, v in enumerate(miss.values):
    ax.text(v + .005, i, f"{v:.0%}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

**읽는 법**: 결측률이 높은 변수(보통 체온)는 병동에서 자주 안 재기 때문입니다.
결측이 50% 넘으면 그 변수 피처는 신뢰도가 낮아요 — 나중에 SHAP에서 기여도가 낮게 나오면
"실제로 안 중요"가 아니라 "데이터가 없어서"일 수 있습니다.

## 1-3. 분포 + 환자별 기록 길이

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, v in zip(axes.ravel(), VITALS):
    ax.hist(cohort.vitals[v].dropna(), bins=50, color="steelblue")
    ax.set_title(v, fontsize=10)
fig.suptitle("Vital-sign distributions (after sanitation)")
plt.tight_layout(); plt.show()

lengths = cohort.vitals.groupby("patient_id")["hour"].max() + 1
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(lengths, bins=40, color="slateblue")
ax[0].set_title("Recorded hours per patient"); ax[0].set_xlabel("hours")
ax[1].bar(["event", "control"], [n_evt, n_pat - n_evt], color=["crimson", "gray"])
ax[1].set_title("Class balance (patient level)")
for i, v in enumerate([n_evt, n_pat - n_evt]):
    ax[1].text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout(); plt.show()

## 1-4. 이벤트 환자의 궤적 — "전조가 실제로 보이는가?"

In [ ]:
evt_ids = cohort.events.loc[cohort.events["arrest_hour"].notna(), "patient_id"].tolist()
pid = evt_ids[0]
g = cohort.vitals[cohort.vitals["patient_id"] == pid].sort_values("hour")
ah = float(cohort.events.set_index("patient_id").loc[pid, "arrest_hour"])

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, v in zip(axes.ravel(), VITALS):
    ax.plot(g["hour"], g[v], marker=".", ms=4)
    ax.axvline(ah, color="red", ls="--", lw=1.5)
    ax.set_title(v, fontsize=10)
fig.suptitle(f"Patient {pid} — trajectory (red = event onset)")
plt.tight_layout(); plt.show()

**읽는 법**: 빨간선(사건 시각) 직전에 맥박↑·SpO₂↓·호흡수↑ 같은 변화가 보이면
"조기경보가 가능하다"는 근거입니다. 한 명만 봐선 모르니 여러 명 보려면 `evt_ids[1]`, `[2]`로 바꿔보세요.
**개인차가 크다는 것**도 확인 포인트 — 그래서 '개인 기저선 이탈' 피처가 필요합니다.

---
# 2단계. Feature Engineering

**목적**: 시계열을 모델이 먹을 수 있는 표(tabular) 형태로 바꾸기.

## 방법: 슬라이딩 윈도우
매 시각 t마다 **과거 8시간**을 보고 → "앞으로 H시간 안에 사건?"을 라벨로 답니다.
각 윈도우에서 vital별로 **평균/표준편차/최소/최대/최근값/기울기/변화량**을 뽑습니다.

## 핵심 차별 피처: 개인 기저선 이탈
같은 맥박 100이라도 평소 60이던 환자면 위험, 평소 95면 정상입니다. 그래서 **환자 본인의
초기 안정기 평균 대비 편차**(`_last_dev`, `_mean_dev`)를 추가합니다. → 개인차성 오경보 감소.

## 2-1. 예측 지평(horizon)이 왜 중요한가 — 실패 사례부터

In [ ]:
from vitals_data import build_windows

rows = []
for H in [1, 2, 3, 6, 12, 24]:
    w = build_windows(cohort, prediction_horizon_hours=H)
    rows.append((H, len(w.labels), int(w.labels.sum()), float(w.labels.mean())))
tab = pd.DataFrame(rows, columns=["horizon_h", "windows", "positives", "positive_rate"])
display(tab.assign(positive_rate=lambda d: (d.positive_rate*100).round(2).astype(str)+"%"))

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(tab.horizon_h, tab.positive_rate*100, marker="o")
ax.set_xlabel("prediction horizon (h)"); ax.set_ylabel("positive rate (%)")
ax.set_title("Wider horizon -> more positives")
plt.tight_layout(); plt.show()

**읽는 법 (중요)**: 지평이 1시간이면 환자당 양성 윈도우가 사실상 1개라 **양성비율 ~0.2%**.
이렇게 희박하면 모델이 배울 게 없어서 AUPRC가 바닥(0.003)으로 붕괴합니다 — 실제로 겪었습니다.
지평 6시간은 "6시간 내 악화"라는 **임상적으로 표준적인 조기경보 프레이밍**이고,
양성이 늘어 학습 가능해집니다. 숫자 조작이 아니라 문제 설정을 올바르게 잡는 것.

## 2-2. 피처 생성 + 환자 단위 분할

In [ ]:
from vitals_data import add_personalized_features, patient_level_split

windowed = add_personalized_features(
    build_windows(cohort, prediction_horizon_hours=HORIZON), cohort
)
print(f"윈도우 {len(windowed.labels):,}개 | 양성 {int(windowed.labels.sum())} "
      f"({windowed.labels.mean():.2%}) | 피처 {windowed.features.shape[1]}개\n")

split = patient_level_split(windowed)
print(f"train {len(split.y_train):,} (양성 {int(split.y_train.sum())}) | "
      f"test {len(split.y_test):,} (양성 {int(split.y_test.sum())})")
BASE_RATE = float(np.mean(split.y_test))
print(f"\n>>> test base rate = {BASE_RATE:.4f}  <-- AUPRC는 이 값과 비교해서 해석!")

print("\n[피처 예시]")
display(windowed.features.head(3).T.head(15))

**환자 단위 분할이 중요한 이유**: 같은 환자의 윈도우가 train/test에 섞이면
모델이 그 환자를 외워버려 성능이 뻥튀기됩니다(데이터 누수). 그래서 환자 통째로 나눕니다.

**base rate를 기억하세요** — 위에서 출력된 값. AUPRC가 이보다 확실히 커야 의미 있습니다.

---
# 3단계. LightGBM / XGBoost / CatBoost 베이스라인

**목적**: 여러 부스팅 모델을 같은 조건에서 비교 + 임상 규칙 **NEWS**와 비교.
NEWS를 이겨야 "AI를 쓸 이유"가 생깁니다.

불균형 처리: `scale_pos_weight = 음성수/양성수`로 양성에 가중치를 줍니다.

In [ ]:
# catboost가 없으면 설치 (선택)
try:
    import catboost; print("catboost", catboost.__version__)
except ImportError:
    print("catboost 미설치 — 쓰려면: pip install catboost  (없어도 나머지는 동작)")

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from vitals_train import evaluate, compute_news_scores

Xtr, ytr = split.X_train, split.y_train
Xte, yte = split.X_test, split.y_test
spw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
print(f"scale_pos_weight = {spw:.1f}")

scores = {}   # 모델명 -> test 예측확률
models = {}

# --- XGBoost ---
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5,
                    subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=spw, eval_metric="aucpr",
                    random_state=SEED, n_jobs=-1)
xgb.fit(Xtr, ytr)
scores["XGBoost"] = xgb.predict_proba(Xte)[:, 1]; models["XGBoost"] = xgb

# --- LightGBM ---
from lightgbm import LGBMClassifier
lgbm = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                      subsample=0.8, colsample_bytree=0.8,
                      scale_pos_weight=spw, random_state=SEED, n_jobs=-1, verbose=-1)
lgbm.fit(Xtr, ytr)
scores["LightGBM"] = lgbm.predict_proba(Xte)[:, 1]; models["LightGBM"] = lgbm

# --- CatBoost (있으면) ---
try:
    from catboost import CatBoostClassifier
    cat = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=5,
                             scale_pos_weight=spw, random_seed=SEED, verbose=0)
    cat.fit(Xtr, ytr)
    scores["CatBoost"] = cat.predict_proba(Xte)[:, 1]; models["CatBoost"] = cat
except ImportError:
    pass

# --- NEWS (임상 규칙 베이스라인) ---
scores["NEWS"] = compute_news_scores(Xte)

print("학습 완료:", list(scores))

In [ ]:
rows = []
for name, s in scores.items():
    m = evaluate(name, yte, s)
    rows.append({"model": name, "AUPRC": m.auprc, "AUPRC/base": m.auprc / BASE_RATE,
                 "ROC": m.roc_auc, "sens@95spec": m.sensitivity_at_95_specificity,
                 "alarms/100": m.alarms_per_100_windows})
board = pd.DataFrame(rows).sort_values("AUPRC", ascending=False).reset_index(drop=True)
print(f"(base rate = {BASE_RATE:.4f} — AUPRC/base가 1.0이면 랜덤과 동일)\n")
display(board.round(3))

**읽는 법**:
- `AUPRC/base`가 **1.0 근처면 랜덤과 다를 바 없음**(실패). 3~10이면 의미 있는 신호.
- **NEWS보다 AUPRC가 높아야** AI를 쓸 근거가 됩니다. 비슷하면 → 데이터가 부족하거나
  피처가 약한 것 (모델 탓이 아님).
- `alarms/100`은 낮을수록 좋음 — 같은 검출률에서 알람이 적어야 임상에서 씁니다.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for name, s in scores.items():
    p, r, _ = precision_recall_curve(yte, s); ax[0].plot(r, p, label=name)
    f, t, _ = roc_curve(yte, s);              ax[1].plot(f, t, label=name)
ax[0].axhline(BASE_RATE, color="gray", ls=":", label=f"base {BASE_RATE:.3f}")
ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision"); ax[0].set_title("PR curve"); ax[0].legend()
ax[1].plot([0,1],[0,1],"k--",alpha=.3)
ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].set_title("ROC curve"); ax[1].legend()
plt.tight_layout(); plt.show()

**PR곡선이 ROC곡선보다 중요한 이유**: 희귀사건에선 ROC가 다 비슷하게 높아 보입니다(함정).
정밀도(=알람이 진짜일 확률)를 보여주는 PR곡선에서 진짜 차이가 드러나요. 회색 점선(base rate)에
붙어있으면 그 모델은 쓸모없는 겁니다.

## 3-1. 피처 중요도 — "무엇이 위험을 알리는가"

In [ ]:
best_name = board.loc[board.model != "NEWS", "model"].iloc[0]
best_model = models[best_name]
imp = pd.Series(best_model.feature_importances_, index=Xtr.columns).nlargest(20)

fig, ax = plt.subplots(figsize=(7, 6))
imp[::-1].plot.barh(ax=ax, color="teal")
ax.set_title(f"Top-20 feature importance ({best_name})")
plt.tight_layout(); plt.show()

n_dev = sum("_dev" in f for f in imp.index)
print(f"상위 20개 중 개인 기저선 이탈(_dev) 피처: {n_dev}개")

**읽는 법**: `_dev`(개인 기저선 이탈) 피처가 상위에 많이 들어있으면 **우리 차별점이
실제로 작동**한다는 증거입니다. 제안서에 쓸 수 있는 근거예요.

---
# 4단계. 하이퍼파라미터 튜닝 및 앙상블

**목적**: (a) Optuna로 최적 파라미터 탐색, (b) 여러 모델을 합쳐 성능·안정성 향상.

**주의**: 튜닝은 **train 안에서 교차검증**으로만 합니다. test를 보고 튜닝하면 반칙(누수)이에요.
환자 단위 GroupKFold를 써서 CV에서도 누수를 막습니다.

## 4-1. Optuna 튜닝

In [ ]:
import optuna
from sklearn.model_selection import GroupKFold
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30      # 시간 없으면 10으로 줄이세요
groups_tr = split.groups_train

def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 800, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
    )
    gkf = GroupKFold(n_splits=4)
    aps = []
    for tr_i, va_i in gkf.split(Xtr, ytr, groups_tr):
        m = XGBClassifier(**params, scale_pos_weight=spw, eval_metric="aucpr",
                          random_state=SEED, n_jobs=-1)
        m.fit(Xtr.iloc[tr_i], ytr[tr_i])          # y_train은 numpy array
        aps.append(average_precision_score(ytr[va_i], m.predict_proba(Xtr.iloc[va_i])[:, 1]))
    return float(np.mean(aps))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print(f"best CV AUPRC = {study.best_value:.4f}")
print("best params:"); display(pd.Series(study.best_params))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
vals = [t.value for t in study.trials if t.value is not None]
ax[0].plot(vals, marker="o", ms=3, alpha=.6)
ax[0].plot(np.maximum.accumulate(vals), color="red", lw=2, label="best so far")
ax[0].set_xlabel("trial"); ax[0].set_ylabel("CV AUPRC"); ax[0].set_title("Optuna optimization history"); ax[0].legend()

imp_p = optuna.importance.get_param_importances(study)
pd.Series(imp_p).plot.barh(ax=ax[1], color="darkorange")
ax[1].set_title("Hyperparameter importance")
plt.tight_layout(); plt.show()

**읽는 법**: 왼쪽 빨간선이 계속 올라가면 튜닝이 효과를 보는 중, 평평해지면 수렴한 것.
오른쪽은 어떤 파라미터가 성능을 좌우하는지 — 다음에 튜닝할 때 그 범위를 집중 탐색하면 됩니다.

## 4-2. 튜닝 모델 + 앙상블

In [ ]:
# 튜닝된 XGBoost
xgb_tuned = XGBClassifier(**study.best_params, scale_pos_weight=spw,
                          eval_metric="aucpr", random_state=SEED, n_jobs=-1)
xgb_tuned.fit(Xtr, ytr)
scores["XGBoost(tuned)"] = xgb_tuned.predict_proba(Xte)[:, 1]

# 앙상블: 각 모델 점수를 순위로 바꿔 평균 (스케일이 달라도 안전한 rank averaging)
from scipy.stats import rankdata
ens_members = [n for n in scores if n not in ("NEWS",)]
ens = np.mean([rankdata(scores[n]) / len(yte) for n in ens_members], axis=0)
scores["Ensemble"] = ens
print("앙상블 구성:", ens_members)

In [ ]:
rows = []
for name, s in scores.items():
    m = evaluate(name, yte, s)
    rows.append({"model": name, "AUPRC": m.auprc, "AUPRC/base": m.auprc / BASE_RATE,
                 "ROC": m.roc_auc, "sens@95spec": m.sensitivity_at_95_specificity,
                 "alarms/100": m.alarms_per_100_windows})
final = pd.DataFrame(rows).sort_values("AUPRC", ascending=False).reset_index(drop=True)
print(f"=== 최종 비교 (base rate {BASE_RATE:.4f}) ===")
display(final.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
d = final.set_index("model")["AUPRC"].sort_values()
colors = ["crimson" if i == "NEWS" else "steelblue" for i in d.index]
d.plot.barh(ax=ax, color=colors)
ax.axvline(BASE_RATE, color="gray", ls=":", label="base rate (random)")
ax.set_xlabel("AUPRC"); ax.set_title("Final model comparison (NEWS = clinical baseline)")
ax.legend(); plt.tight_layout(); plt.show()

---
# 결과 해석 가이드

위 최종 표를 보고 아래 순서로 판단하세요.

### ① AUPRC/base 가 1에 가까운가?
→ **그렇다면 모델이 아무것도 못 배운 것**. 원인은 보통:
- 데이터가 너무 적음 (환자 수 늘리기: `MAX_FILES=None`)
- 지평이 너무 좁음 (`HORIZON` 키우기)
- 활력징후만으론 예측이 어려운 사건 (패혈증은 lab 수치가 중요)

### ② 부스팅 모델이 NEWS를 이기는가?
- **이긴다** → 우리 주장("AI가 규칙보다 오경보 적다")의 실데이터 근거 확보 ✅
- **비슷하다** → 아직 근거 부족. 데이터를 늘려서 재확인. 모델을 더 복잡하게 하는 건 답이 아님.

### ③ alarms/100 이 NEWS보다 낮은가?
→ 이게 우리 **핵심 차별점**입니다. 같은 검출률에서 알람이 적어야 임상에서 씁니다.

### ④ _dev 피처가 중요도 상위에 있는가?
→ '개인 기저선 이탈'이라는 novelty가 실제로 작동한다는 증거.

---

## 다음에 해볼 것
1. `MAX_FILES=None`으로 전체 데이터 (가장 효과 큼)
2. `HORIZON`을 3/6/12로 바꿔 비교
3. `N_TRIALS`를 50~100으로 늘려 튜닝 강화
4. SHAP으로 개별 경보 설명: `python src/vitals_explain.py`

> 다시 강조: 이 숫자는 **패혈증 프록시** 결과입니다. 심정지 성능이 아니며, 제안서에는
> 합성 데이터 시연 수치로 정직하게 표기되어 있습니다.